In [145]:
# 판다스 넘파이 가져오기
import pandas as pd
import numpy as np

In [146]:
# 파일 불러오기
order = pd.read_csv('./data/orders.csv')
oi = pd.read_csv('./data/order_items.csv')
prod = pd.read_csv('./data/products.csv')
cust = pd.read_csv('./data/customers.csv')

In [147]:
prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    int64  
 1   product_name  500 non-null    str    
 2   category      500 non-null    str    
 3   price         500 non-null    str    
 4   cost          500 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.5 KB


In [148]:
# orders 파일 정제
print(f'주문일시 결측 주문 건수 : {order["order_datetime"].isna().sum()}')
orders=order.dropna(subset='order_datetime')
print(f'주문일시 결측 제거 후 주문 건수 : {orders.shape[0]}')
orders=orders.drop_duplicates(subset='order_id')
print(f'중복 제거 후 주문건 수 : {orders.shape[0]}')
orders['order_datetime']=pd.to_datetime(orders['order_datetime'],errors="coerce")
orders.info()

주문일시 결측 주문 건수 : 1933
주문일시 결측 제거 후 주문 건수 : 198067
중복 제거 후 주문건 수 : 198067
<class 'pandas.DataFrame'>
Index: 198067 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        198067 non-null  int64         
 1   customer_id     198067 non-null  int64         
 2   order_datetime  198067 non-null  datetime64[us]
 3   channel         198067 non-null  str           
 4   status          198067 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 11.2 MB


In [149]:
#product함수 관련 결측치 및 제거
prod=prod.drop_duplicates()
prod['price']=pd.to_numeric(prod['price'].astype(str)
                            .str.replace(',','').str.replace('원',''))
prod= prod[prod['price']>0]
prod= prod[prod['cost']>0]
prod.info()

<class 'pandas.DataFrame'>
Index: 489 entries, 1 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    489 non-null    int64  
 1   product_name  489 non-null    str    
 2   category      489 non-null    str    
 3   price         489 non-null    float64
 4   cost          489 non-null    float64
dtypes: float64(2), int64(1), str(2)
memory usage: 35.2 KB


In [150]:
# order_item 파일 정제 (1, 주문 아이템 아이디 중복제거)
orderitem=oi.drop_duplicates(subset='order_item_id')
orderitem.info()

<class 'pandas.DataFrame'>
Index: 499880 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  499880 non-null  int64  
 1   order_id       499880 non-null  int64  
 2   product_id     499880 non-null  int64  
 3   quantity       499880 non-null  int64  
 4   unit_price     484774 non-null  float64
 5   discount       499880 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 26.7 MB


In [151]:
# order_item 파일 정제 (2. 유닛 프라이스 채우기)
orderitem= orderitem.merge(prod[['product_id','price']],how='left',on='product_id')
orderitem['unit_price']= orderitem['unit_price'].fillna(orderitem["price"])

orderitem=orderitem.drop_duplicates()
orderitem=orderitem.drop(columns='price')
orderitem.info()

<class 'pandas.DataFrame'>
RangeIndex: 499880 entries, 0 to 499879
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  499880 non-null  int64  
 1   order_id       499880 non-null  int64  
 2   product_id     499880 non-null  int64  
 3   quantity       499880 non-null  int64  
 4   unit_price     499641 non-null  float64
 5   discount       499880 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 22.9 MB


In [152]:
# customer 파일 정제
cust.info()
cust.head()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  5000 non-null   int64
 1   name         5000 non-null   str  
 2   gender       4696 non-null   str  
 3   birth_date   4616 non-null   str  
 4   signup_date  5000 non-null   str  
 5   city         5000 non-null   str  
 6   email        4752 non-null   str  
dtypes: int64(1), str(6)
memory usage: 544.3 KB


,customer_id,name,gender,birth_date,signup_date,city,email
0,3293,권준서,여,1954-04-29,2020-04-26,부산,user3293@example.com
1,1862,박준선,F,1980-05-19,2023-09-01,서울,user1862@example.com
2,4955,전영경,여,1980-06-08,2024-04-30,인천,user4955@example.com
3,4653,한재하,M,1971-06-09,2023-08-06,서울,user4653@example.com
4,3331,송준연,여,1971-02-01,2023-12-20,대전,user3331@example.com


In [153]:
#중복치 제거
cust=cust.drop_duplicates(subset='customer_id')
#birth_date, signup_date 정제
cust['birth_date']=pd.to_datetime(cust['birth_date'],errors="coerce")
cust['signup_date']=pd.to_datetime(cust['signup_date'],errors="coerce")
# city 정제
cust['city']=cust['city'].str.strip()
#gender 정제
gmap={'여':'F','F':'F','남':'M','M':'M'}
cust['gender']=cust['gender'].map(gmap)
#결측치는 빼지 않고, 일단 냅두기로 함. 데이터에 사용가치가 있음.

In [154]:
cust.info()

<class 'pandas.DataFrame'>
Index: 4950 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  4950 non-null   int64         
 1   name         4950 non-null   str           
 2   gender       4649 non-null   str           
 3   birth_date   4572 non-null   datetime64[us]
 4   signup_date  4950 non-null   datetime64[us]
 5   city         4950 non-null   str           
 6   email        4702 non-null   str           
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 479.4 KB


In [155]:
# orders & order_items 결합
mix_ooi = orderitem.merge(orders,how='left',on='order_id',validate="m:1")
mix_ooi.info()

<class 'pandas.DataFrame'>
RangeIndex: 499880 entries, 0 to 499879
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   499880 non-null  int64         
 1   order_id        499880 non-null  int64         
 2   product_id      499880 non-null  int64         
 3   quantity        499880 non-null  int64         
 4   unit_price      499641 non-null  float64       
 5   discount        499880 non-null  float64       
 6   customer_id     495100 non-null  float64       
 7   order_datetime  495100 non-null  datetime64[us]
 8   channel         495100 non-null  str           
 9   status          495100 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(4), str(2)
memory usage: 43.5 MB


In [156]:
#주문정보가 존재하지 않는, 주문 품목 갯수 발견, 정확한 주문과 주문일시를 확인이 불가능하므로,
# 제거 예정
print(f'주문에 들어가있지 않은 주문 품목 수 : {mix_ooi["order_datetime"].isna().sum()}')
mix_ooi = mix_ooi.dropna(subset='order_datetime')
print(f'결측치 제거 후 총 주문품목수 : {mix_ooi.shape[0]}')

주문에 들어가있지 않은 주문 품목 수 : 4780
결측치 제거 후 총 주문품목수 : 495100


In [157]:
# 순매출을 구하기 위해서 prod와 병합(price 제외)
mixed = mix_ooi.merge(prod[['product_id','product_name','category','cost']],how='left',on='product_id',validate="m:1")
mixed.info()

<class 'pandas.DataFrame'>
RangeIndex: 495100 entries, 0 to 495099
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   495100 non-null  int64         
 1   order_id        495100 non-null  int64         
 2   product_id      495100 non-null  int64         
 3   quantity        495100 non-null  int64         
 4   unit_price      494862 non-null  float64       
 5   discount        495100 non-null  float64       
 6   customer_id     495100 non-null  float64       
 7   order_datetime  495100 non-null  datetime64[us]
 8   channel         495100 non-null  str           
 9   status          495100 non-null  str           
 10  product_name    487853 non-null  str           
 11  category        487853 non-null  str           
 12  cost            487853 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(4), str(4)
memory usage: 66.3 MB


In [158]:
cust_ids = set(cust['customer_id'])
order_ids = set(mixed['customer_id'])

print('고객 원장 고객 수:', len(cust_ids))
print('주문 데이터 고객 수:', len(order_ids))
print('주문에만 있는 고객 수:', len(order_ids - cust_ids))
print('고객 원장에만 있는 고객 수:', len(cust_ids - order_ids))

#주문데이터에만 있는 고객들은 존재, 고객원장에만 있는 고객들은 존재하지 않음.
#일단 VFM모집군에 주문데이터만 있는 고객들만 처리하기엔 애매한 상황이므로 포함시키지 않을 예정.

고객 원장 고객 수: 4950
주문 데이터 고객 수: 5858
주문에만 있는 고객 수: 908
고객 원장에만 있는 고객 수: 0


In [159]:
# cust 병합(cust위주)
mixed= cust.merge(mixed,how="left",on="customer_id",validate="1:m")
mixed.info()
mixed['customer_id'].nunique()

<class 'pandas.DataFrame'>
RangeIndex: 492642 entries, 0 to 492641
Data columns (total 19 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   customer_id     492642 non-null  int64         
 1   name            492642 non-null  str           
 2   gender          457770 non-null  str           
 3   birth_date      464088 non-null  datetime64[us]
 4   signup_date     492642 non-null  datetime64[us]
 5   city            492642 non-null  str           
 6   email           477434 non-null  str           
 7   order_item_id   492642 non-null  int64         
 8   order_id        492642 non-null  int64         
 9   product_id      492642 non-null  int64         
 10  quantity        492642 non-null  int64         
 11  unit_price      492406 non-null  float64       
 12  discount        492642 non-null  float64       
 13  order_datetime  492642 non-null  datetime64[us]
 14  channel         492642 non-null  str           

4950

In [160]:
mixed['customer_id'].nunique()

4950

In [161]:
mixed['amount'] = (mixed['quantity'] * mixed['unit_price'])
mixed['p_amount'] = (mixed['quantity'] * mixed['unit_price'])*(1-mixed['discount'])
# 취소 환불 제거.
mixed = mixed[~(mixed['status'].isin(['canceled','returned']))]
mixed['status'].unique()
# 2024년 6월까지 데이터 제거
mixed=mixed[mixed['order_datetime']<'2024-07-01']

In [162]:
mixed=mixed.set_index('customer_id')

In [163]:
amount_month = mixed.groupby('customer_id').agg(순매출=("p_amount","sum")
                                        ,유효주문수=("order_id","nunique")
                                        ,최근주문일=("order_datetime","max"))

amount_month['마지막구매경과일'] = (pd.Timestamp('2024-07-01') - amount_month['최근주문일']).dt.days


In [170]:
amount_month['R']= pd.qcut(amount_month['마지막구매경과일'],q=5,labels=[5,4,3,2,1]).astype('int16')
amount_month['F']= pd.qcut(amount_month['유효주문수'],q=5,labels=[1,2,3,4,5]).astype('int16')
amount_month['M']= pd.qcut(amount_month['순매출'],q=5,labels=[1,2,3,4,5]).astype('int16')


#### 등급 분류 기준

다음과 같이 처리할 예정 (순서는 R(마지막구매 경과일) / F(구매 빈도수) / M(순매출))

챔피언 5,5,5

충성 4,4,4

잠재충성 4,3,3

일반 3,2,2

이탈 위험 2,1,1

휴면 1,1,1  

In [171]:


amount_month['회원등급']='6.휴면'

amount_month.loc[(amount_month['R']>=2) &
                (amount_month['F']>=1) &
                (amount_month['M']>=1),
                '회원등급'
                ]='5.이탈위험'

amount_month.loc[(amount_month['R']>=3) &
                (amount_month['F']>=2) &
                (amount_month['M']>=2),
                '회원등급'
                ]='4.일반'

amount_month.loc[(amount_month['R']>=4) &
                (amount_month['F']>=3) &
                (amount_month['M']>=3),
                '회원등급'
                ]='3.잠재충성'

amount_month.loc[(amount_month['R']>=4) &
                (amount_month['F']>=4) &
                (amount_month['M']>=4),
                '회원등급'
                ]='2.충성'

amount_month.loc[(amount_month['R']==5) &
                (amount_month['F']==5) &
                (amount_month['M']==5),
                '회원등급'
                ]='1.챔피언'


In [172]:
amount_month

,순매출,유효주문수,최근주문일,마지막구매경과일,R,F,M,회원등급
customer_id,,,,,,,,
1000,2530766.0,7,2024-06-20 18:13:10,10,2,1,2,5.이탈위험
1001,6223603.0,22,2024-06-23 06:40:41,7,3,4,4,4.일반
1002,2432546.0,9,2024-06-15 05:28:16,15,2,2,2,5.이탈위험
1003,18893715.0,56,2024-06-29 15:02:30,1,5,5,5,1.챔피언
1004,7910790.0,23,2024-06-29 06:50:00,1,5,4,4,2.충성
...,...,...,...,...,...,...,...,...
5945,5277713.0,17,2024-06-13 11:47:19,17,1,4,4,6.휴면
5946,9569031.0,22,2024-06-25 19:12:14,5,3,4,5,4.일반
5947,16238112.0,61,2024-06-27 22:54:50,3,4,5,5,2.충성


In [176]:
segment = amount_month.groupby("회원등급").agg(
    고객수=("회원등급", "count"),
    매출=("순매출", "sum")
)

In [177]:
segment["고객비중(%)"] = (
    segment["고객수"]
    / segment["고객수"].sum()
    * 100
).round(2)

segment["매출비중(%)"] = (
    segment["매출"]
    / segment["매출"].sum()
    * 100
).round(2)

segment.sort_index()

,고객수,매출,고객비중(%),매출비중(%)
회원등급,,,,
1.챔피언,613,2.724932e+10,12.38,57.01
2.충성,562,5.198877e+09,11.35,10.88
3.잠재충성,295,1.401300e+09,5.96,2.93
4.일반,1175,6.545533e+09,23.74,13.69
5.이탈위험,1352,4.615898e+09,27.31,9.66
6.휴면,953,2.784540e+09,19.25,5.83
